# Electrostatic quadrupole

`BeamOptics` on a 3D grid: a 10 keV proton beam through an electrostatic quadrupole, the
case of IBSimu's `sim_3d_quad.py`. Four plates, 20 mm long, sit 8.1 mm from the axis: the
pair across $y$ at +2 kV, the pair across $z$ at -2 kV. The protons are pushed towards the
axis in $y$ and away from it in $z$, so the lens focuses in one plane and defocuses in the
other.

The grid is `GridParams(ndim=3)`, the electrodes are 3D shapes (`cuboid`), the faces of
`FieldBCs` include $z$, and the beam is a `BeamSpecies3D`: trajectories on a square
perpendicular to $x$.

In [ ]:
import plasma_plots as pp

from beamtracer import (
    BeamOptics,
    BeamOpticsConfig,
    FieldBCs,
    FieldBoundaryCondition,
    GridParams,
    ObjectParams,
)
from beamtracer.config import BeamSpecies3D

BLUE, ORANGE = "C0", "C1"
pp.set_backend("plotly")  # interactive figures, embedded in the docs
pp.set_math("latex")  # formulas in the figures typeset as LaTeX

## The lens

The mesh has 1 mm cells over 100 mm by 20 mm by 20 mm. The faces have a zero normal
derivative, so the potential is that of the plates alone. The plates' faces lie between
nodes.

In [ ]:
def plate(axis: int, sign: int, potential: float) -> ObjectParams:
    centre, sides = [0.0503, 0.0, 0.0], [0.02, 0.03, 0.03]
    centre[axis] = sign * 0.0116
    sides[axis] = 0.007
    return ObjectParams(
        type="cuboid",
        midpoint=tuple(centre),
        side_lengths=tuple(sides),
        bc_type="dirichlet",
        bc_value=potential,
    )


neumann = FieldBoundaryCondition.neumann(0.0)
config = BeamOpticsConfig(
    grid=GridParams(
        ndim=3,
        num_gridpoints=(101, 21, 21),
        lower_bound=(0.0, -0.01, -0.01),
        upper_bound=(0.1, 0.01, 0.01),
    ),
    objects={
        "y+": plate(1, 1, 2000.0),
        "y-": plate(1, -1, 2000.0),
        "z+": plate(2, 1, -2000.0),
        "z-": plate(2, -1, -2000.0),
    },
    bcs_field=FieldBCs(*([neumann] * 6)),
    beams=[
        BeamSpecies3D(
            "H+",
            10.0,
            centre=(0.0, 0.0, 0.0),
            size=(0.004, 0.004),
            energy=10e3,
            transverse_temperature=0.0,
            num_particles=900,
        )
    ],
    iterations=2,
    tolerance=0.0,
    diagnostic_planes=[0.095],
    save_trajectories=30,
)
model = BeamOptics(config).run()
print(model.stats)

## The beam at the exit

The moments at $x = 95$ mm, in each transverse plane. The beam leaves smaller in $y$ than
in $z$, where part of it reaches the faces of the mesh.

In [ ]:
for axis in ("y", "z"):
    m = model.beam_moments(0.095, axis=axis)
    print(
        f"{axis}: rms size {m.rms_size * 1e3:.2f} mm, rms angle {m.rms_angle * 1e3:.1f} mrad, "
        f"emittance {m.emittance * 1e6:.3f} mm mrad"
    )

In [ ]:
hits = model.plane(0.095)
with pp.figure(title="The beam at x = 95 mm") as fig:
    ax = fig[0]
    ax.scatter(hits.point[1] * 1e3, hits.point[2] * 1e3, s=6, color=BLUE)
    ax.set_xlabel("$y$ [mm]")
    ax.set_ylabel("$z$ [mm]")
    ax.set_aspect("equal")

## The potential and the trajectories

The potential in the plane $z = 0$, and the kept trajectories projected on it: they are
squeezed in $y$ by the lens.

In [ ]:
ds = model.dataset()
middle = ds.sel(z=0.0, method="nearest")
with pp.figure(title="Potential at z = 0") as fig:
    ax = fig[0]
    mesh = ax.pcolormesh(
        ds.x * 1e3, ds.y * 1e3, middle.potential.T, cmap="RdBu_r", shading="auto"
    )
    ax.figure.colorbar(mesh, ax=ax, label=r"$\phi$ [V]")
    for path in model.trajectories:
        ax.plot(path[:, 0] * 1e3, path[:, 1] * 1e3, color="k", lw=0.6)
    ax.set_xlabel("$x$ [mm]")
    ax.set_ylabel("$y$ [mm]")